<div style="border-top:4px solid #0f766e;padding:28px 0 18px">
<div style="color:#0f766e;font-size:13px;font-weight:700;letter-spacing:.8px">LAB 06 · LEVEL 2 · JOINING DATA</div>
<div style="color:#17212b;font-size:30px;font-weight:750">Join facts and dimensions, then read the plan</div>
<p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px">Use a controlled fact sample to compare inner, left, semi, and anti join semantics. Then inspect the distributed plan and relate the chosen data movement to table sizes.</p>
<span style="display:inline-block;border:1px solid #99f6e4;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:10px;font-size:12px">Prerequisite: Lab 4 · this lab owns its derived tables</span>
</div>

## Join contract

The `join_events_lab6` table is a small fact sample with one deliberately missing product. The dimension has one row per product key. Keeping the sample deterministic makes row multiplication and unmatched-row behavior easy to verify.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")


In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS join_events_lab6 (
    event_id BIGINT NOT NULL,
    product_id BIGINT NOT NULL,
    region VARCHAR(16) NOT NULL,
    revenue DECIMAL(12,2) NOT NULL
)
DUPLICATE KEY(event_id)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.execute("TRUNCATE TABLE join_events_lab6")
lab.insert("""
INSERT INTO join_events_lab6 VALUES
    (610001, 1005115, 'region_01', 29.95),
    (610002, 1005115, 'region_02', 15.00),
    (610003, 13200021, 'region_01', 9.99),
    (610004, 99999999, 'region_03', 12.50)
""", title="Load controlled fact sample")

lab.execute("""
CREATE TABLE IF NOT EXISTS dim_products_lab6 (
    product_id BIGINT NOT NULL,
    category VARCHAR(32) NOT NULL,
    brand VARCHAR(32) NOT NULL
)
UNIQUE KEY(product_id)
DISTRIBUTED BY HASH(product_id) BUCKETS 1
PROPERTIES ("replication_num"="1", "enable_unique_key_merge_on_write"="true")
""")
lab.execute("TRUNCATE TABLE dim_products_lab6")
lab.insert("""
INSERT INTO dim_products_lab6 VALUES
    (1005115, 'electronics', 'doris-demo'),
    (13200021, 'home', 'doris-demo'),
    (77777777, 'outdoor', 'doris-demo')
""", title="Load one-row-per-key dimension")

In [ ]:
lab.sql("""
SELECT f.event_id, f.product_id, f.region, d.category, d.brand
FROM join_events_lab6 AS f
INNER JOIN dim_products_lab6 AS d ON d.product_id = f.product_id
ORDER BY f.event_id
""", title="Inner join keeps matched facts")

lab.sql("""
SELECT f.event_id, f.product_id, d.category
FROM join_events_lab6 AS f
LEFT JOIN dim_products_lab6 AS d ON d.product_id = f.product_id
ORDER BY f.event_id
""", title="Left join keeps the orphan fact")

## Existence joins and row-grain checks

Semi and anti joins answer existence questions without returning dimension columns. The row-count comparison makes a duplicate dimension key visible rather than hiding it in a larger result.

In [ ]:
lab.sql("""
SELECT f.event_id, f.product_id
FROM join_events_lab6 AS f
WHERE EXISTS (
    SELECT 1 FROM dim_products_lab6 AS d
    WHERE d.product_id = f.product_id
)
ORDER BY f.event_id
""", title="Semi-join: facts with a product definition")

lab.sql("""
SELECT f.event_id, f.product_id
FROM join_events_lab6 AS f
WHERE NOT EXISTS (
    SELECT 1 FROM dim_products_lab6 AS d
    WHERE d.product_id = f.product_id
)
ORDER BY f.event_id
""", title="Anti-join: orphan product IDs")

lab.sql("""
SELECT
    (SELECT COUNT(*) FROM join_events_lab6) AS fact_rows,
    (SELECT COUNT(*) FROM (
        SELECT f.event_id
        FROM join_events_lab6 AS f
        INNER JOIN dim_products_lab6 AS d ON d.product_id = f.product_id
    ) AS matched) AS matched_rows,
    (SELECT COUNT(*) FROM (
        SELECT f.event_id
        FROM join_events_lab6 AS f
        LEFT JOIN dim_products_lab6 AS d ON d.product_id = f.product_id
    ) AS preserved) AS left_join_rows
""", title="Join row-count contract")

## Read the distributed plan

On a single-node sandbox, elapsed time is not evidence of a join strategy. `EXPLAIN` is the evidence: look for the join type, hash condition, and whether Doris broadcasts the small dimension or shuffles both inputs.

In [ ]:
lab.sql("""
EXPLAIN
SELECT f.region, d.category, SUM(f.revenue) AS revenue
FROM join_events_lab6 AS f
LEFT JOIN dim_products_lab6 AS d ON d.product_id = f.product_id
GROUP BY f.region, d.category
ORDER BY f.region, d.category
""", title="Explain the fact-dimension plan", final=True)

## Takeaway

- Inner joins remove unmatched facts; left joins preserve the left-hand grain.
- Semi and anti joins are existence tests, not alternate ways to project dimension columns.
- A one-to-many dimension can multiply facts. Validate key uniqueness before aggregating.
- Use plan evidence to explain data movement; do not infer strategy from a tiny local runtime.